# Vectors and Spaces

Companion notebook for the [Vectors and Spaces](https://ml-viz.vercel.app/courses/linear-algebra/01-vectors-and-spaces) lesson.

We'll build intuition for vectors, norms, dot products, and orthogonality using NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#30344a',
    'axes.labelcolor':  '#e2e8f0',
    'xtick.color':      '#94a3b8',
    'ytick.color':      '#94a3b8',
    'text.color':       '#e2e8f0',
    'grid.color':       '#30344a',
})

## Vector operations

In [ ]:
u = np.array([2.0, 1.0])
v = np.array([1.0, 3.0])

print('u + v =', u + v)
print('2 * u =', 2 * u)
print('dot product u·v =', np.dot(u, v))
print('‖u‖ =', np.linalg.norm(u))
print('‖v‖ =', np.linalg.norm(v))

cos_theta = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
angle_deg = np.degrees(np.arccos(np.clip(cos_theta, -1, 1)))
print(f'Angle between u and v: {angle_deg:.1f}°')

## Visualizing vectors and their angle

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: vectors in 2D space
ax = axes[0]
origin = np.array([0, 0])
colors = ['#6366f1', '#2dd4bf', '#f97316']
vectors = [u, v, u + v]
labels = ['u = [2,1]', 'v = [1,3]', 'u+v']

for vec, color, label in zip(vectors, colors, labels):
    ax.annotate('', xy=vec, xytext=origin,
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    ax.text(vec[0] * 1.05, vec[1] * 1.05, label, color=color, fontsize=11)

# Parallelogram for addition
para = plt.Polygon([origin, u, u+v, v], fill=False,
                   edgecolor='#6366f1', alpha=0.3, linestyle='--')
ax.add_patch(para)
ax.set_xlim(-0.5, 4); ax.set_ylim(-0.5, 5)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title('Vector Addition', pad=12)

# Right: dot product and orthogonality
ax2 = axes[1]
a = np.array([1.0, 0.0])
b = np.array([0.0, 1.0])
c = np.array([1.0, 1.0]) / np.sqrt(2)

for vec, color, label in [(a, '#6366f1', 'a'), (b, '#2dd4bf', 'b'),
                          (c * 2, '#f97316', 'c (45°)')]:
    ax2.annotate('', xy=vec, xytext=[0,0],
                 arrowprops=dict(arrowstyle='->', color=color, lw=2))
    ax2.text(vec[0]*1.1, vec[1]*1.1, f'{label}\ndot={np.dot(a,vec):.2f}',
             color=color, fontsize=10)

ax2.set_xlim(-0.3, 1.8); ax2.set_ylim(-0.3, 1.8)
ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)
ax2.set_title('Dot Products vs. a=[1,0]', pad=12)

plt.tight_layout()
plt.show()

## Norms and normalization

The Euclidean norm $\|\mathbf{v}\| = \sqrt{v_1^2 + v_2^2}$ measures the length of a vector.
Dividing a vector by its norm produces a **unit vector** with direction but no magnitude.

In [ ]:
vectors_to_normalize = [
    np.array([3.0, 4.0]),
    np.array([1.0, 1.0]),
    np.array([5.0, 0.0]),
]

for v in vectors_to_normalize:
    norm = np.linalg.norm(v)
    unit = v / norm
    print(f'v={v}  ‖v‖={norm:.3f}  unit={unit.round(3)}  ‖unit‖={np.linalg.norm(unit):.4f}')

## Cosine similarity in practice

Cosine similarity is the dot product of **unit** vectors. It's used for word embeddings,
document similarity, and recommendation systems.

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Simulated word embeddings (toy example)
embeddings = {
    'king':   np.array([0.9, 0.8, 0.1, 0.0]),
    'queen':  np.array([0.8, 0.9, 0.2, 0.0]),
    'man':    np.array([0.7, 0.1, 0.0, 0.0]),
    'woman':  np.array([0.6, 0.2, 0.0, 0.0]),
    'apple':  np.array([0.0, 0.0, 0.8, 0.9]),
}

words = list(embeddings.keys())
for i, w1 in enumerate(words):
    for w2 in words[i+1:]:
        sim = cosine_similarity(embeddings[w1], embeddings[w2])
        if sim > 0.9:
            print(f'{w1:8s} ↔ {w2:8s}: similarity = {sim:.3f}')

## Projection — the dot product's geometric job

The dot product measures *how much of $\mathbf{b}$ points along $\mathbf{a}$*. Starting from $\mathbf{a}\cdot\mathbf{b} = \|\mathbf{a}\|\,\|\mathbf{b}\|\cos\theta$:

$$
\text{scalar proj} = \|\mathbf{b}\|\cos\theta = \frac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|},
\qquad
\operatorname{proj}_{\mathbf{a}}\mathbf{b} = \frac{\mathbf{a}\cdot\mathbf{b}}{\mathbf{a}\cdot\mathbf{a}}\,\mathbf{a}.
$$

The residual $\mathbf{b} - \operatorname{proj}_{\mathbf{a}}\mathbf{b}$ is **orthogonal** to $\mathbf{a}$ — the property that makes least-squares regression and PCA work.

In [ ]:
a = np.array([3.0, 1.0])
b = np.array([1.0, 2.0])

# scalar projection = ||b|| cos(theta) = (a·b) / ||a||
scalar_proj = np.dot(a, b) / np.linalg.norm(a)
# vector projection = (a·b)/(a·a) * a   -- lands on the line through a
vector_proj = (np.dot(a, b) / np.dot(a, a)) * a
residual = b - vector_proj

print(f'scalar projection of b onto a : {scalar_proj:.3f}')
print(f'vector projection             : {vector_proj.round(3)}')
print(f'residual  b - proj            : {residual.round(3)}')

# Key fact used by least squares: the residual is ORTHOGONAL to a
print(f'residual · a = {np.dot(residual, a):.2e}  (=> 0, so residual ⟂ a)')

## Linear independence, span, and rank

Stack vectors as the **columns** of a matrix. The **rank** is the number of independent directions among them:

- rank = number of vectors  ⇒  **independent** — they span an $n$-dimensional subspace and form a basis for it.
- rank < number of vectors  ⇒  **dependent** — at least one is a linear combination of the others and adds no new direction.

In ML, a rank-deficient feature matrix means redundant features (perfect multicollinearity), which makes the normal equations singular and models unstable.

In [ ]:
def describe(vectors, name):
    M = np.column_stack(vectors)
    r = np.linalg.matrix_rank(M)
    n = M.shape[1]
    status = f'independent (spans R^{M.shape[0]})' if r == n else 'DEPENDENT'
    print(f'{name}: rank {r} of {n} columns -> {status}')

describe([np.array([1, 2]), np.array([2, 4])], 'a=[1,2], b=[2,4]')   # b = 2a -> dependent
describe([np.array([1, 2]), np.array([2, 1])], 'a=[1,2], c=[2,1]')   # independent

# With an independent basis we can express ANY target as a unique combination:
# solve M c = target  for the coefficients c
M = np.column_stack([np.array([1.0, 2.0]), np.array([2.0, 1.0])])
target = np.array([4.0, 5.0])
coeffs = np.linalg.solve(M, target)
print(f'\n[4,5] = {coeffs[0]:.2f}*[1,2] + {coeffs[1]:.2f}*[2,1]')
print('check:', (coeffs[0] * M[:, 0] + coeffs[1] * M[:, 1]).round(3))